# RC/SPA Tests (1d U17 + 1d U12 + 4h U12)

This notebook runs RC/SPA tests on precomputed WF OOS returns and writes outputs to `results/rc_spa_tests/`.


In [1]:
# Cell 1: Setup + Paths + Params
from __future__ import annotations

import importlib.util
import logging
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

from src.artifacts import safe_write_parquet
from src.rc_spa import run_rc_spa_returns, run_rc_spa_utility
from src import stats_tests as st_module

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("rc_spa_tests_nb")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(f"Cannot locate project root from cwd={Path.cwd()}")

RESULTS_DIR = PROJECT_ROOT / "results" / "rc_spa_tests"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

INPUTS = {
    "base_1d_returns": PROJECT_ROOT / "results" / "runner_exp" / "1d" / "returns_oos.parquet",
    "base_1d_bench": PROJECT_ROOT / "results" / "runner_exp" / "1d" / "bench_returns_oos.parquet",
    "onchain_1d_returns": PROJECT_ROOT / "results" / "runner_onchain" / "returns_oos.parquet",
    "base_4h_returns": PROJECT_ROOT / "results" / "runner_exp" / "4h" / "returns_oos.parquet",
    "base_4h_bench": PROJECT_ROOT / "results" / "runner_exp" / "4h" / "bench_returns_oos.parquet",
}

SYMBOLS_1D = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "XRPUSDT", "DOGEUSDT"]
SYMBOLS_4H = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "XRPUSDT", "DOGEUSDT"]
COSTS = ["Low", "Base", "High"]

RC_PARAMS = {
    "reps": 2000,
    "seed": 42,
    "studentize": True,
    "alpha": 0.05,
}
LAMBDAS = [0.5, 1.0, 2.0]
BLOCK_SIZE = {"1d": 10, "4h": 60}

for name, path in INPUTS.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing required input: {name} -> {path}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("arch installed in current kernel:", bool(importlib.util.find_spec("arch")))
print("inputs:")
for name, path in INPUTS.items():
    print(f"  - {name}: {path}")


PROJECT_ROOT: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma
RESULTS_DIR: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\rc_spa_tests
arch installed in current kernel: True
inputs:
  - base_1d_returns: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\runner_exp\1d\returns_oos.parquet
  - base_1d_bench: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\runner_exp\1d\bench_returns_oos.parquet
  - onchain_1d_returns: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\runner_onchain\returns_oos.parquet
  - base_4h_returns: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\runner_exp\4h\returns_oos.parquet
  - base_4h_bench: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\runner_exp\4h\bench_returns_oos.parquet


In [2]:
# Cell 2: Load + Sanity

def ensure_date_col(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()
    if "date" not in df.columns:
        if "datetime_utc" in df.columns:
            df = df.rename(columns={"datetime_utc": "date"})
        elif "index" in df.columns:
            df = df.rename(columns={"index": "date"})
        else:
            raise KeyError(f"No date-like column found. Columns: {list(df.columns)}")
    df["date"] = pd.to_datetime(df["date"])
    return df


def assert_unique_keys(df: pd.DataFrame, *, name: str) -> None:
    key_cols = ["date", "symbol", "cost", "strategy_id"]
    dups = int(df.duplicated(subset=key_cols).sum())
    if dups > 0:
        raise RuntimeError(f"{name}: found duplicate rows by {key_cols}: {dups}")


def coverage_snapshot(df: pd.DataFrame, *, name: str) -> pd.DataFrame:
    out = pd.DataFrame(
        {
            "dataset": [name],
            "rows": [len(df)],
            "symbols": [df["symbol"].astype(str).nunique()],
            "costs": [df["cost"].astype(str).nunique()],
            "strategies": [df["strategy_id"].astype(str).nunique()],
            "min_date": [pd.to_datetime(df["date"]).min()],
            "max_date": [pd.to_datetime(df["date"]).max()],
        }
    )
    return out


base_1d_returns = ensure_date_col(pd.read_parquet(INPUTS["base_1d_returns"]))
base_1d_bench = ensure_date_col(pd.read_parquet(INPUTS["base_1d_bench"]))
onchain_1d_returns = ensure_date_col(pd.read_parquet(INPUTS["onchain_1d_returns"]))
base_4h_returns = ensure_date_col(pd.read_parquet(INPUTS["base_4h_returns"]))
base_4h_bench = ensure_date_col(pd.read_parquet(INPUTS["base_4h_bench"]))

assert_unique_keys(base_1d_returns, name="base_1d_returns")
assert_unique_keys(base_1d_bench, name="base_1d_bench")
assert_unique_keys(onchain_1d_returns, name="onchain_1d_returns")
assert_unique_keys(base_4h_returns, name="base_4h_returns")
assert_unique_keys(base_4h_bench, name="base_4h_bench")

snapshots = pd.concat(
    [
        coverage_snapshot(base_1d_returns, name="base_1d_returns"),
        coverage_snapshot(base_1d_bench, name="base_1d_bench"),
        coverage_snapshot(onchain_1d_returns, name="onchain_1d_returns"),
        coverage_snapshot(base_4h_returns, name="base_4h_returns"),
        coverage_snapshot(base_4h_bench, name="base_4h_bench"),
    ],
    ignore_index=True,
)

display(snapshots)

base_1d_nsid = base_1d_returns["strategy_id"].astype(str).nunique()
onchain_1d_nsid = onchain_1d_returns["strategy_id"].astype(str).nunique()
base_4h_nsid = base_4h_returns["strategy_id"].astype(str).nunique()

print("base_1d strategy count:", base_1d_nsid)
print("onchain_1d strategy count:", onchain_1d_nsid)
print("base_4h strategy count:", base_4h_nsid)

if base_1d_nsid != 12:
    raise AssertionError(f"Expected 12 base strategies on 1d, got {base_1d_nsid}")
if onchain_1d_nsid != 5:
    raise AssertionError(f"Expected 5 onchain strategies on 1d, got {onchain_1d_nsid}")
if base_4h_nsid != 12:
    raise AssertionError(f"Expected 12 base strategies on 4h, got {base_4h_nsid}")

print("Sanity checks passed.")


,dataset,rows,symbols,costs,strategies,min_date,max_date
0,base_1d_returns,302472,5,3,12,2020-09-16 00:00:00,2026-02-03
1,base_1d_bench,25206,5,3,1,2020-09-16 00:00:00,2026-02-03
2,onchain_1d_returns,98640,4,3,5,2020-09-16 00:00:00,2026-02-03
3,base_4h_returns,1894536,5,3,12,2020-08-22 04:00:00,2026-02-22
4,base_4h_bench,157878,5,3,1,2020-08-22 04:00:00,2026-02-22


base_1d strategy count: 12
onchain_1d strategy count: 5
base_4h strategy count: 12
Sanity checks passed.


In [3]:
# Cell 3: 1d U17 (base + onchain)

returns_u17_1d = pd.concat([base_1d_returns, onchain_1d_returns], ignore_index=True)
assert_unique_keys(returns_u17_1d, name="returns_u17_1d")
bench_u17_1d = base_1d_bench.copy()

rc_returns_1d_u17 = run_rc_spa_returns(
    symbols=SYMBOLS_1D,
    costs=COSTS,
    returns_oos_long=returns_u17_1d,
    bench_long=bench_u17_1d,
    st_module=st_module,
    block_size=int(BLOCK_SIZE["1d"]),
    reps=int(RC_PARAMS["reps"]),
    seed=int(RC_PARAMS["seed"]),
    studentize=bool(RC_PARAMS["studentize"]),
    alpha=float(RC_PARAMS["alpha"]),
    logger=logger,
)
rc_returns_1d_u17["timeframe"] = "1d"
rc_returns_1d_u17["scenario"] = "u17"

rc_utility_1d_u17 = run_rc_spa_utility(
    symbols=SYMBOLS_1D,
    costs=COSTS,
    returns_oos_long=returns_u17_1d,
    bench_long=bench_u17_1d,
    st_module=st_module,
    lambdas=LAMBDAS,
    block_size=int(BLOCK_SIZE["1d"]),
    reps=int(RC_PARAMS["reps"]),
    seed=int(RC_PARAMS["seed"]),
    studentize=bool(RC_PARAMS["studentize"]),
    alpha=float(RC_PARAMS["alpha"]),
    logger=logger,
)
rc_utility_1d_u17["timeframe"] = "1d"
rc_utility_1d_u17["scenario"] = "u17"

safe_write_parquet(rc_returns_1d_u17, RESULTS_DIR / "rc_spa_returns_1d_u17.parquet")
safe_write_parquet(rc_utility_1d_u17, RESULTS_DIR / "rc_spa_utility_1d_u17.parquet")

print("rc_returns_1d_u17:", rc_returns_1d_u17.shape)
print("rc_utility_1d_u17:", rc_utility_1d_u17.shape)
display(rc_returns_1d_u17.head(10))
display(rc_utility_1d_u17.head(10))


2026-04-06 14:52:14,426 | INFO | RC/SPA universe: BTCUSDT | Low | T=1826 | models=17
2026-04-06 14:52:20,874 | INFO | RC/SPA universe: BTCUSDT | Base | T=1826 | models=17
2026-04-06 14:52:25,498 | INFO | RC/SPA universe: BTCUSDT | High | T=1826 | models=17
2026-04-06 14:52:30,074 | INFO | RC/SPA universe: ETHUSDT | Low | T=1826 | models=17
2026-04-06 14:52:34,576 | INFO | RC/SPA universe: ETHUSDT | Base | T=1826 | models=17
2026-04-06 14:52:39,115 | INFO | RC/SPA universe: ETHUSDT | High | T=1826 | models=17
2026-04-06 14:52:43,535 | INFO | RC/SPA universe: BNBUSDT | Low | T=1826 | models=12
2026-04-06 14:52:47,974 | INFO | RC/SPA universe: BNBUSDT | Base | T=1826 | models=12
2026-04-06 14:52:52,458 | INFO | RC/SPA universe: BNBUSDT | High | T=1826 | models=12
2026-04-06 14:52:56,955 | INFO | RC/SPA universe: XRPUSDT | Low | T=1644 | models=17
2026-04-06 14:53:01,370 | INFO | RC/SPA universe: XRPUSDT | Base | T=1644 | models=17
2026-04-06 14:53:05,421 | INFO | RC/SPA universe: XRPUSDT 

rc_returns_1d_u17: (15, 16)
rc_utility_1d_u17: (45, 17)


,symbol,cost,T,universe_size,rc_pvalue,rc_stat,rc_best_model,spa_pvalue_lower,spa_pvalue_consistent,spa_pvalue_upper,reps,block_size,studentize,spa_n_better_models,timeframe,scenario
0,BTCUSDT,Low,1826,17,0.950025,-0.000261,T2:EMA_Crossover,0.6095,0.9165,0.9525,2000.0,10.0,1.0,0.0,1d,u17
1,BTCUSDT,Base,1826,17,0.953023,-0.000266,T2:EMA_Crossover,0.6300,0.9180,0.9530,2000.0,10.0,1.0,0.0,1d,u17
2,BTCUSDT,High,1826,17,0.960520,-0.000308,T2:EMA_Crossover,0.6490,0.9355,0.9650,2000.0,10.0,1.0,0.0,1d,u17
3,ETHUSDT,Low,1826,17,0.856572,-0.000014,S5:Ensemble_3Signals,0.5930,0.8400,0.8545,2000.0,10.0,1.0,0.0,1d,u17
4,ETHUSDT,Base,1826,17,0.857571,-0.000017,S5:Ensemble_3Signals,0.5930,0.8400,0.8545,2000.0,10.0,1.0,0.0,1d,u17
5,ETHUSDT,High,1826,17,0.858571,-0.000022,S5:Ensemble_3Signals,0.5905,0.8420,0.8570,2000.0,10.0,1.0,0.0,1d,u17
6,BNBUSDT,Low,1826,12,0.962519,-0.000226,T2:EMA_Crossover,0.6150,0.9335,0.9495,2000.0,10.0,1.0,0.0,1d,u17
7,BNBUSDT,Base,1826,12,0.963018,-0.000232,T2:EMA_Crossover,0.6125,0.9355,0.9505,2000.0,10.0,1.0,0.0,1d,u17
8,BNBUSDT,High,1826,12,0.936032,-0.000141,T1:SMA_Crossover,0.5775,0.9115,0.9335,2000.0,10.0,1.0,0.0,1d,u17
9,XRPUSDT,Low,1644,17,0.838581,0.000097,R2:Bollinger_MR,0.6285,0.8335,0.8490,2000.0,10.0,1.0,0.0,1d,u17


,symbol,cost,lam,T,universe_size,rc_pvalue,rc_stat,rc_best_model,spa_pvalue_lower,spa_pvalue_consistent,spa_pvalue_upper,reps,block_size,studentize,spa_n_better_models,timeframe,scenario
0,BTCUSDT,Low,0.5,1826,17,0.115442,0.001325,S3:Breakout_Confirm_MA,0.1280,0.1280,0.1280,2000.0,10.0,1.0,0.0,1d,u17
1,BTCUSDT,Low,1.0,1826,17,0.000500,0.004303,S1OC:MAFilter_RSI_MR_Onchain,0.0000,0.0000,0.0000,2000.0,10.0,1.0,13.0,1d,u17
2,BTCUSDT,Low,2.0,1826,17,0.000500,0.010283,S1OC:MAFilter_RSI_MR_Onchain,0.0000,0.0000,0.0000,2000.0,10.0,1.0,17.0,1d,u17
3,BTCUSDT,Base,0.5,1826,17,0.114943,0.001322,S3:Breakout_Confirm_MA,0.1280,0.1280,0.1280,2000.0,10.0,1.0,0.0,1d,u17
4,BTCUSDT,Base,1.0,1826,17,0.000500,0.004300,S1OC:MAFilter_RSI_MR_Onchain,0.0000,0.0000,0.0000,2000.0,10.0,1.0,13.0,1d,u17
5,BTCUSDT,Base,2.0,1826,17,0.000500,0.010277,S1OC:MAFilter_RSI_MR_Onchain,0.0000,0.0000,0.0000,2000.0,10.0,1.0,17.0,1d,u17
6,BTCUSDT,High,0.5,1826,17,0.122439,0.001316,S3:Breakout_Confirm_MA,0.1290,0.1290,0.1290,2000.0,10.0,1.0,0.0,1d,u17
7,BTCUSDT,High,1.0,1826,17,0.000500,0.004292,S1OC:MAFilter_RSI_MR_Onchain,0.0000,0.0000,0.0000,2000.0,10.0,1.0,13.0,1d,u17
8,BTCUSDT,High,2.0,1826,17,0.000500,0.010263,S1OC:MAFilter_RSI_MR_Onchain,0.0000,0.0000,0.0000,2000.0,10.0,1.0,17.0,1d,u17
9,ETHUSDT,Low,0.5,1826,17,0.051974,0.002128,R2OC:Bollinger_MR_Onchain,0.0605,0.0605,0.0605,2000.0,10.0,1.0,0.0,1d,u17


In [4]:
# Cell 4: 1d U12 (base only)

returns_u12_1d = base_1d_returns.copy()
bench_u12_1d = base_1d_bench.copy()

rc_returns_1d_u12 = run_rc_spa_returns(
    symbols=SYMBOLS_1D,
    costs=COSTS,
    returns_oos_long=returns_u12_1d,
    bench_long=bench_u12_1d,
    st_module=st_module,
    block_size=int(BLOCK_SIZE["1d"]),
    reps=int(RC_PARAMS["reps"]),
    seed=int(RC_PARAMS["seed"]),
    studentize=bool(RC_PARAMS["studentize"]),
    alpha=float(RC_PARAMS["alpha"]),
    logger=logger,
)
rc_returns_1d_u12["timeframe"] = "1d"
rc_returns_1d_u12["scenario"] = "u12"

rc_utility_1d_u12 = run_rc_spa_utility(
    symbols=SYMBOLS_1D,
    costs=COSTS,
    returns_oos_long=returns_u12_1d,
    bench_long=bench_u12_1d,
    st_module=st_module,
    lambdas=LAMBDAS,
    block_size=int(BLOCK_SIZE["1d"]),
    reps=int(RC_PARAMS["reps"]),
    seed=int(RC_PARAMS["seed"]),
    studentize=bool(RC_PARAMS["studentize"]),
    alpha=float(RC_PARAMS["alpha"]),
    logger=logger,
)
rc_utility_1d_u12["timeframe"] = "1d"
rc_utility_1d_u12["scenario"] = "u12"

safe_write_parquet(rc_returns_1d_u12, RESULTS_DIR / "rc_spa_returns_1d_u12.parquet")
safe_write_parquet(rc_utility_1d_u12, RESULTS_DIR / "rc_spa_utility_1d_u12.parquet")

print("rc_returns_1d_u12:", rc_returns_1d_u12.shape)
print("rc_utility_1d_u12:", rc_utility_1d_u12.shape)
display(rc_returns_1d_u12.head(10))
display(rc_utility_1d_u12.head(10))


2026-04-06 14:57:57,062 | INFO | RC/SPA universe: BTCUSDT | Low | T=1826 | models=12
2026-04-06 14:58:01,677 | INFO | RC/SPA universe: BTCUSDT | Base | T=1826 | models=12
2026-04-06 14:58:06,262 | INFO | RC/SPA universe: BTCUSDT | High | T=1826 | models=12
2026-04-06 14:58:10,795 | INFO | RC/SPA universe: ETHUSDT | Low | T=1826 | models=12
2026-04-06 14:58:15,320 | INFO | RC/SPA universe: ETHUSDT | Base | T=1826 | models=12
2026-04-06 14:58:19,791 | INFO | RC/SPA universe: ETHUSDT | High | T=1826 | models=12
2026-04-06 14:58:24,344 | INFO | RC/SPA universe: BNBUSDT | Low | T=1826 | models=12
2026-04-06 14:58:29,119 | INFO | RC/SPA universe: BNBUSDT | Base | T=1826 | models=12
2026-04-06 14:58:33,924 | INFO | RC/SPA universe: BNBUSDT | High | T=1826 | models=12
2026-04-06 14:58:38,695 | INFO | RC/SPA universe: XRPUSDT | Low | T=1644 | models=12
2026-04-06 14:58:42,844 | INFO | RC/SPA universe: XRPUSDT | Base | T=1644 | models=12
2026-04-06 14:58:46,930 | INFO | RC/SPA universe: XRPUSDT 

rc_returns_1d_u12: (15, 16)
rc_utility_1d_u12: (45, 17)


,symbol,cost,T,universe_size,rc_pvalue,rc_stat,rc_best_model,spa_pvalue_lower,spa_pvalue_consistent,spa_pvalue_upper,reps,block_size,studentize,spa_n_better_models,timeframe,scenario
0,BTCUSDT,Low,1826,12,0.949025,-0.000261,T2:EMA_Crossover,0.6090,0.9165,0.9515,2000.0,10.0,1.0,0.0,1d,u12
1,BTCUSDT,Base,1826,12,0.952024,-0.000266,T2:EMA_Crossover,0.6295,0.9180,0.9520,2000.0,10.0,1.0,0.0,1d,u12
2,BTCUSDT,High,1826,12,0.959020,-0.000308,T2:EMA_Crossover,0.6490,0.9355,0.9650,2000.0,10.0,1.0,0.0,1d,u12
3,ETHUSDT,Low,1826,12,0.853573,-0.000014,S5:Ensemble_3Signals,0.5865,0.8305,0.8500,2000.0,10.0,1.0,0.0,1d,u12
4,ETHUSDT,Base,1826,12,0.854573,-0.000017,S5:Ensemble_3Signals,0.5865,0.8310,0.8500,2000.0,10.0,1.0,0.0,1d,u12
5,ETHUSDT,High,1826,12,0.855572,-0.000022,S5:Ensemble_3Signals,0.5835,0.8330,0.8530,2000.0,10.0,1.0,0.0,1d,u12
6,BNBUSDT,Low,1826,12,0.962519,-0.000226,T2:EMA_Crossover,0.6150,0.9335,0.9495,2000.0,10.0,1.0,0.0,1d,u12
7,BNBUSDT,Base,1826,12,0.963018,-0.000232,T2:EMA_Crossover,0.6125,0.9355,0.9505,2000.0,10.0,1.0,0.0,1d,u12
8,BNBUSDT,High,1826,12,0.936032,-0.000141,T1:SMA_Crossover,0.5775,0.9115,0.9335,2000.0,10.0,1.0,0.0,1d,u12
9,XRPUSDT,Low,1644,12,0.834083,0.000097,R2:Bollinger_MR,0.6070,0.8305,0.8465,2000.0,10.0,1.0,0.0,1d,u12


,symbol,cost,lam,T,universe_size,rc_pvalue,rc_stat,rc_best_model,spa_pvalue_lower,spa_pvalue_consistent,spa_pvalue_upper,reps,block_size,studentize,spa_n_better_models,timeframe,scenario
0,BTCUSDT,Low,0.5,1826,12,0.106447,0.001325,S3:Breakout_Confirm_MA,0.1160,0.1160,0.1160,2000.0,10.0,1.0,0.0,1d,u12
1,BTCUSDT,Low,1.0,1826,12,0.000500,0.003921,S2:MA200Filter_Bollinger_MR,0.0000,0.0000,0.0000,2000.0,10.0,1.0,9.0,1d,u12
2,BTCUSDT,Low,2.0,1826,12,0.000500,0.009556,S2:MA200Filter_Bollinger_MR,0.0000,0.0000,0.0000,2000.0,10.0,1.0,12.0,1d,u12
3,BTCUSDT,Base,0.5,1826,12,0.104948,0.001322,S3:Breakout_Confirm_MA,0.1165,0.1165,0.1165,2000.0,10.0,1.0,0.0,1d,u12
4,BTCUSDT,Base,1.0,1826,12,0.000500,0.003918,S2:MA200Filter_Bollinger_MR,0.0000,0.0000,0.0000,2000.0,10.0,1.0,9.0,1d,u12
5,BTCUSDT,Base,2.0,1826,12,0.000500,0.009553,S2:MA200Filter_Bollinger_MR,0.0000,0.0000,0.0000,2000.0,10.0,1.0,12.0,1d,u12
6,BTCUSDT,High,0.5,1826,12,0.112444,0.001316,S3:Breakout_Confirm_MA,0.1195,0.1195,0.1195,2000.0,10.0,1.0,0.0,1d,u12
7,BTCUSDT,High,1.0,1826,12,0.000500,0.003912,S2:MA200Filter_Bollinger_MR,0.0000,0.0000,0.0000,2000.0,10.0,1.0,9.0,1d,u12
8,BTCUSDT,High,2.0,1826,12,0.000500,0.009546,S2:MA200Filter_Bollinger_MR,0.0000,0.0000,0.0000,2000.0,10.0,1.0,12.0,1d,u12
9,ETHUSDT,Low,0.5,1826,12,0.069965,0.001921,S2:MA200Filter_Bollinger_MR,0.0760,0.0760,0.0760,2000.0,10.0,1.0,0.0,1d,u12


In [5]:
# Cell 5: 4h U12 (base only)

returns_u12_4h = base_4h_returns.copy()
bench_u12_4h = base_4h_bench.copy()

rc_returns_4h_u12 = run_rc_spa_returns(
    symbols=SYMBOLS_4H,
    costs=COSTS,
    returns_oos_long=returns_u12_4h,
    bench_long=bench_u12_4h,
    st_module=st_module,
    block_size=int(BLOCK_SIZE["4h"]),
    reps=int(RC_PARAMS["reps"]),
    seed=int(RC_PARAMS["seed"]),
    studentize=bool(RC_PARAMS["studentize"]),
    alpha=float(RC_PARAMS["alpha"]),
    logger=logger,
)
rc_returns_4h_u12["timeframe"] = "4h"
rc_returns_4h_u12["scenario"] = "u12"

rc_utility_4h_u12 = run_rc_spa_utility(
    symbols=SYMBOLS_4H,
    costs=COSTS,
    returns_oos_long=returns_u12_4h,
    bench_long=bench_u12_4h,
    st_module=st_module,
    lambdas=LAMBDAS,
    block_size=int(BLOCK_SIZE["4h"]),
    reps=int(RC_PARAMS["reps"]),
    seed=int(RC_PARAMS["seed"]),
    studentize=bool(RC_PARAMS["studentize"]),
    alpha=float(RC_PARAMS["alpha"]),
    logger=logger,
)
rc_utility_4h_u12["timeframe"] = "4h"
rc_utility_4h_u12["scenario"] = "u12"

safe_write_parquet(rc_returns_4h_u12, RESULTS_DIR / "rc_spa_returns_4h_u12.parquet")
safe_write_parquet(rc_utility_4h_u12, RESULTS_DIR / "rc_spa_utility_4h_u12.parquet")

print("rc_returns_4h_u12:", rc_returns_4h_u12.shape)
print("rc_utility_4h_u12:", rc_utility_4h_u12.shape)
display(rc_returns_4h_u12.head(10))
display(rc_utility_4h_u12.head(10))


2026-04-06 15:03:05,463 | INFO | RC/SPA universe: BTCUSDT | Low | T=12060 | models=12
2026-04-06 15:03:32,556 | INFO | RC/SPA universe: BTCUSDT | Base | T=12060 | models=12
2026-04-06 15:04:02,694 | INFO | RC/SPA universe: BTCUSDT | High | T=12060 | models=12
2026-04-06 15:04:31,331 | INFO | RC/SPA universe: ETHUSDT | Low | T=12060 | models=12
2026-04-06 15:05:00,142 | INFO | RC/SPA universe: ETHUSDT | Base | T=12060 | models=12
2026-04-06 15:05:28,725 | INFO | RC/SPA universe: ETHUSDT | High | T=12060 | models=12
2026-04-06 15:05:57,305 | INFO | RC/SPA universe: BNBUSDT | Low | T=10956 | models=12
2026-04-06 15:06:21,794 | INFO | RC/SPA universe: BNBUSDT | Base | T=10956 | models=12
2026-04-06 15:06:46,657 | INFO | RC/SPA universe: BNBUSDT | High | T=10956 | models=12
2026-04-06 15:07:10,912 | INFO | RC/SPA universe: XRPUSDT | Low | T=9870 | models=12
2026-04-06 15:07:31,611 | INFO | RC/SPA universe: XRPUSDT | Base | T=9870 | models=12
2026-04-06 15:07:52,522 | INFO | RC/SPA universe:

rc_returns_4h_u12: (15, 16)
rc_utility_4h_u12: (45, 17)


,symbol,cost,T,universe_size,rc_pvalue,rc_stat,rc_best_model,spa_pvalue_lower,spa_pvalue_consistent,spa_pvalue_upper,reps,block_size,studentize,spa_n_better_models,timeframe,scenario
0,BTCUSDT,Low,12060,12,0.888556,-0.000010,S5:Ensemble_3Signals,0.6495,0.8275,0.8795,2000.0,60.0,1.0,0.0,4h,u12
1,BTCUSDT,Base,12060,12,0.894053,-0.000011,S5:Ensemble_3Signals,0.6505,0.8265,0.8865,2000.0,60.0,1.0,0.0,4h,u12
2,BTCUSDT,High,12060,12,0.901549,-0.000012,S5:Ensemble_3Signals,0.6330,0.8395,0.8945,2000.0,60.0,1.0,0.0,4h,u12
3,ETHUSDT,Low,12060,12,0.849075,0.000009,T2:EMA_Crossover,0.6840,0.8235,0.8315,2000.0,60.0,1.0,0.0,4h,u12
4,ETHUSDT,Base,12060,12,0.853573,0.000006,S5:Ensemble_3Signals,0.6875,0.8315,0.8360,2000.0,60.0,1.0,0.0,4h,u12
5,ETHUSDT,High,12060,12,0.861069,0.000004,S5:Ensemble_3Signals,0.6480,0.8365,0.8400,2000.0,60.0,1.0,0.0,4h,u12
6,BNBUSDT,Low,10956,12,0.991504,-0.000071,T2:EMA_Crossover,0.6345,0.9890,0.9895,2000.0,60.0,1.0,0.0,4h,u12
7,BNBUSDT,Base,10956,12,0.995002,-0.000075,T2:EMA_Crossover,0.6400,0.9910,0.9915,2000.0,60.0,1.0,0.0,4h,u12
8,BNBUSDT,High,10956,12,0.995502,-0.000084,T2:EMA_Crossover,0.6275,0.9890,0.9940,2000.0,60.0,1.0,0.0,4h,u12
9,XRPUSDT,Low,9870,12,0.909545,-0.000025,S3:Breakout_Confirm_MA,0.6455,0.8900,0.9125,2000.0,60.0,1.0,0.0,4h,u12


,symbol,cost,lam,T,universe_size,rc_pvalue,rc_stat,rc_best_model,spa_pvalue_lower,spa_pvalue_consistent,spa_pvalue_upper,reps,block_size,studentize,spa_n_better_models,timeframe,scenario
0,BTCUSDT,Low,0.5,12060,12,0.0005,0.000872,R1:RSI_MR,0.0,0.0,0.0,2000.0,60.0,1.0,12.0,4h,u12
1,BTCUSDT,Low,1.0,12060,12,0.0005,0.001982,R1:RSI_MR,0.0,0.0,0.0,2000.0,60.0,1.0,12.0,4h,u12
2,BTCUSDT,Low,2.0,12060,12,0.0005,0.004203,R1:RSI_MR,0.0,0.0,0.0,2000.0,60.0,1.0,12.0,4h,u12
3,BTCUSDT,Base,0.5,12060,12,0.0005,0.000878,R1:RSI_MR,0.0,0.0,0.0,2000.0,60.0,1.0,12.0,4h,u12
4,BTCUSDT,Base,1.0,12060,12,0.0005,0.001978,R1:RSI_MR,0.0,0.0,0.0,2000.0,60.0,1.0,12.0,4h,u12
5,BTCUSDT,Base,2.0,12060,12,0.0005,0.004196,S2:MA200Filter_Bollinger_MR,0.0,0.0,0.0,2000.0,60.0,1.0,12.0,4h,u12
6,BTCUSDT,High,0.5,12060,12,0.0005,0.000875,R1:RSI_MR,0.0,0.0,0.0,2000.0,60.0,1.0,12.0,4h,u12
7,BTCUSDT,High,1.0,12060,12,0.0005,0.001985,S2:MA200Filter_Bollinger_MR,0.0,0.0,0.0,2000.0,60.0,1.0,12.0,4h,u12
8,BTCUSDT,High,2.0,12060,12,0.0005,0.004227,S2:MA200Filter_Bollinger_MR,0.0,0.0,0.0,2000.0,60.0,1.0,12.0,4h,u12
9,ETHUSDT,Low,0.5,12060,12,0.0005,0.001079,S2:MA200Filter_Bollinger_MR,0.0,0.0,0.0,2000.0,60.0,1.0,12.0,4h,u12


In [6]:
# Cell 6: Compare Views + Save

def check_expected_rows(df: pd.DataFrame, *, expected_rows: int, name: str) -> None:
    got = int(len(df))
    if got != int(expected_rows):
        raise AssertionError(f"{name}: expected {expected_rows} rows, got {got}")


def assert_universe_sizes(df: pd.DataFrame, *, expected: dict[tuple[str, str], int], name: str) -> None:
    for (symbol, cost), exp_sz in expected.items():
        g = df[(df["symbol"] == symbol) & (df["cost"] == cost)]
        if g.empty:
            raise AssertionError(f"{name}: missing ({symbol}, {cost})")
        got = int(g.iloc[0]["universe_size"])
        if got != int(exp_sz):
            raise AssertionError(f"{name}: ({symbol}, {cost}) expected universe_size={exp_sz}, got {got}")


check_expected_rows(rc_returns_1d_u17, expected_rows=15, name="rc_returns_1d_u17")
check_expected_rows(rc_returns_1d_u12, expected_rows=15, name="rc_returns_1d_u12")
check_expected_rows(rc_returns_4h_u12, expected_rows=15, name="rc_returns_4h_u12")
check_expected_rows(rc_utility_1d_u17, expected_rows=45, name="rc_utility_1d_u17")
check_expected_rows(rc_utility_1d_u12, expected_rows=45, name="rc_utility_1d_u12")
check_expected_rows(rc_utility_4h_u12, expected_rows=45, name="rc_utility_4h_u12")

expected_u12_all = {(s, c): 12 for s in SYMBOLS_1D for c in COSTS}
assert_universe_sizes(rc_returns_1d_u12, expected=expected_u12_all, name="rc_returns_1d_u12")
assert_universe_sizes(rc_returns_4h_u12, expected=expected_u12_all, name="rc_returns_4h_u12")

expected_u17_1d = {}
for s in SYMBOLS_1D:
    for c in COSTS:
        expected_u17_1d[(s, c)] = 12 if s == "BNBUSDT" else 17
assert_universe_sizes(rc_returns_1d_u17, expected=expected_u17_1d, name="rc_returns_1d_u17")

returns_all = pd.concat([rc_returns_1d_u17, rc_returns_1d_u12, rc_returns_4h_u12], ignore_index=True)
utility_all = pd.concat([rc_utility_1d_u17, rc_utility_1d_u12, rc_utility_4h_u12], ignore_index=True)

returns_view = (
    returns_all
    .groupby(["timeframe", "scenario"], as_index=False)
    .agg(
        groups=("symbol", "count"),
        rc_pvalue_median=("rc_pvalue", "median"),
        rc_pvalue_max=("rc_pvalue", "max"),
        spa_consistent_median=("spa_pvalue_consistent", "median"),
        universe_min=("universe_size", "min"),
        universe_max=("universe_size", "max"),
    )
    .sort_values(["timeframe", "scenario"])
)

utility_view = (
    utility_all
    .groupby(["timeframe", "scenario", "lam"], as_index=False)
    .agg(
        groups=("symbol", "count"),
        rc_pvalue_median=("rc_pvalue", "median"),
        spa_consistent_median=("spa_pvalue_consistent", "median"),
        universe_min=("universe_size", "min"),
        universe_max=("universe_size", "max"),
    )
    .sort_values(["timeframe", "scenario", "lam"])
)

safe_write_parquet(returns_view, RESULTS_DIR / "rc_spa_returns_compare_view.parquet")
safe_write_parquet(utility_view, RESULTS_DIR / "rc_spa_utility_compare_view.parquet")

print("Saved outputs to:", RESULTS_DIR)
display(returns_view)
display(utility_view)


Saved outputs to: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\rc_spa_tests


,timeframe,scenario,groups,rc_pvalue_median,rc_pvalue_max,spa_consistent_median,universe_min,universe_max
0,1d,u12,15,0.854573,0.963018,0.8335,12,12
1,1d,u17,15,0.857571,0.963018,0.8410,12,17
2,4h,u12,15,0.911044,0.995502,0.8940,12,12


,timeframe,scenario,lam,groups,rc_pvalue_median,spa_consistent_median,universe_min,universe_max
0,1d,u12,0.5,15,0.070465,0.077,12,12
1,1d,u12,1.0,15,0.000500,0.000,12,12
2,1d,u12,2.0,15,0.000500,0.000,12,12
3,1d,u17,0.5,15,0.052474,0.062,12,17
4,1d,u17,1.0,15,0.000500,0.000,12,17
5,1d,u17,2.0,15,0.000500,0.000,12,17
6,4h,u12,0.5,15,0.000500,0.000,12,12
7,4h,u12,1.0,15,0.000500,0.000,12,12
8,4h,u12,2.0,15,0.000500,0.000,12,12
